# 00 — Pure Python fluency (no NumPy)

**Goal:** find out whether you can write Python on your own, without Claude, and fix it if not. Everything here is
plain Python: comprehensions, sorting with keys, recursion, classes, generators, exceptions, dataclasses. Each exercise
is a small piece of ML plumbing (vocabularies, batchers, a bigram language model, a patient-level split).
**Time:** ~2–3 h. **Level:** 1 / 5.

**Rules for every notebook in `04-practice/`**
- You write every line. A function gives you its signature, a docstring and sometimes hints; the body is yours.
- No Claude / Copilot / autocomplete writing code. Turn inline suggestions off in your editor. Claude Code is allowed
  only for the hint ladder (hint → bigger hint → micro-example), after 15 min stuck.
- Fill the `✍️ Predict` cells **before** running, with a confidence tag `[1]` guessing / `[2]` fairly sure / `[3]` certain.
- Run the `check` cell. ✅ = done. ❌ says what is wrong, not how to fix it.
- Note the time each part took. That number is the honest measure of your Python level.

In [1]:
import random, math, re
from dataclasses import dataclass, field

def check(cond, msg):
    assert cond, "❌ " + msg
    print("✅", msg)

def raises(exc, fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except exc:
        return True
    except Exception:
        return False
    return False

## Part 1 — Comprehensions, sorting, recursion (30 min)

No imports beyond what the setup cell has. `collections.Counter` is **not** allowed in this part.

In [2]:
def word_counts(text):
    """Lowercase, split on anything that is not a letter a-z, drop empty strings, count. Returns a dict.
    Hint: re.split(r"[^a-z]+", ...) or str methods."""
    counts = {}
    for w in re.split(r"[^a-z]+", text.lower()):
        if w:
            counts[w] = counts.get(w, 0) + 1
    return counts

def top_k(counts, k):
    """The k (word, count) pairs with the highest count; ties broken alphabetically. One sorted() call with a key."""
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:k]

def transpose(matrix):
    """List of lists (n rows, m cols) -> (m rows, n cols). One line with zip."""
    return [list(row) for row in zip(*matrix)]

def flatten(nested):
    """Flatten arbitrarily nested lists: [1, [2, [3, [4]]], 5] -> [1, 2, 3, 4, 5]. Recursive."""
    out = []
    for x in nested:
        out.extend(flatten(x) if isinstance(x, list) else [x])
    return out

In [3]:
text = "The patient slept. The patient woke up; the nurse slept!"
check(word_counts(text) == {"the": 3, "patient": 2, "slept": 2, "woke": 1, "up": 1, "nurse": 1}, "word_counts")
check(top_k(word_counts(text), 3) == [("the", 3), ("patient", 2), ("slept", 2)], "top_k with alphabetical ties")
check(transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]], "transpose")
check(flatten([1, [2, [3, [4]]], 5, []]) == [1, 2, 3, 4, 5], "flatten")

✅ word_counts
✅ top_k with alphabetical ties
✅ transpose
✅ flatten


## Part 2 — Pure-Python linear algebra and exceptions (20 min)

In [4]:
def matmul(A, B):
    """Matrix product of lists of lists. Raise ValueError with a message containing both shapes if they don't match.
    No NumPy. Use a nested comprehension for the result."""
    n, k, k2, m = len(A), len(A[0]), len(B), len(B[0])
    if k != k2:
        raise ValueError(f"shapes ({n}, {k}) and ({k2}, {m}) do not align")
    return [[sum(A[i][t] * B[t][j] for t in range(k)) for j in range(m)] for i in range(n)]

def softmax(z):
    """Stable softmax of a list of floats, returns a list. Must work for [1000, 1000]."""
    mx = max(z)
    e = [math.exp(v - mx) for v in z]
    s = sum(e)
    return [v / s for v in e]

In [5]:
check(matmul([[1, 2], [3, 4]], [[5], [6]]) == [[17], [39]], "matmul")
check(raises(ValueError, matmul, [[1, 2]], [[1, 2]]), "matmul raises ValueError on bad shapes")
try:
    matmul([[1, 2, 3]], [[1], [2]])
except ValueError as e:
    check("3" in str(e) and "2" in str(e), "error message mentions the shapes")
p = softmax([1000.0, 1000.0, 1000.0])
check(all(abs(v - 1 / 3) < 1e-12 for v in p), "softmax is stable")

✅ matmul
✅ matmul raises ValueError on bad shapes
✅ error message mentions the shapes
✅ softmax is stable


## Part 3 — A class: `Vocab` (30 min)

Every NLP or sequence model starts with this: map tokens to integer ids and back.

In [6]:
class Vocab:
    """Token <-> id mapping.
    - id 0 is "<pad>", id 1 is "<unk>", always present.
    - Other tokens with frequency >= min_freq get ids 2, 3, ... ordered by decreasing frequency, ties alphabetical.
    - encode() maps unknown tokens to 1; decode() maps ids back (ids out of range -> "<unk>").
    - len(vocab) and `token in vocab` must work (dunder methods)."""

    PAD, UNK = "<pad>", "<unk>"

    def __init__(self, tokens, min_freq=1):
        counts = {}
        for t in tokens:
            counts[t] = counts.get(t, 0) + 1
        kept = sorted((t for t, c in counts.items() if c >= min_freq), key=lambda t: (-counts[t], t))
        self.itos = [self.PAD, self.UNK] + kept
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def __contains__(self, token):
        return token in self.stoi

    def encode(self, tokens):
        return [self.stoi.get(t, 1) for t in tokens]

    def decode(self, ids):
        return [self.itos[i] if 0 <= i < len(self.itos) else self.UNK for i in ids]

In [7]:
toks = "b a c a b a d".split()
v = Vocab(toks, min_freq=2)
check(len(v) == 4 and v.itos == ["<pad>", "<unk>", "a", "b"], "vocab order: specials, then by frequency")
check("a" in v and "d" not in v, "__contains__")
check(v.encode(["a", "d", "b"]) == [2, 1, 3], "encode maps unknown to 1")
check(v.decode([3, 2, 99]) == ["b", "a", "<unk>"], "decode")

✅ vocab order: specials, then by frequency
✅ __contains__
✅ encode maps unknown to 1
✅ decode


## Part 4 — Generators: batches and sliding windows (20 min)

A generator `yield`s items one at a time instead of building a list: this is how data loaders stream data that does
not fit in memory.

In [8]:
def batches(data, batch_size, shuffle=False, seed=0, drop_last=False):
    """Yield lists of at most batch_size items. If shuffle, use random.Random(seed) on a COPY (never mutate data).
    If drop_last, skip a final incomplete batch."""
    idx = list(range(len(data)))
    if shuffle:
        random.Random(seed).shuffle(idx)
    for s in range(0, len(idx), batch_size):
        chunk = idx[s:s + batch_size]
        if drop_last and len(chunk) < batch_size:
            return
        yield [data[i] for i in chunk]

def sliding_windows(seq, size, step=1):
    """Yield consecutive windows (as lists) of length `size`, starting every `step` elements; no partial windows."""
    for s in range(0, len(seq) - size + 1, step):
        yield list(seq[s:s + size])

In [9]:
import types
data = list(range(10))
check(isinstance(batches(data, 3), types.GeneratorType), "batches is a generator (uses yield)")
check(list(batches(data, 4)) == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9]], "batches")
check(list(batches(data, 4, drop_last=True)) == [[0, 1, 2, 3], [4, 5, 6, 7]], "drop_last")
sh = list(batches(data, 4, shuffle=True, seed=1))
check(sorted(x for b in sh for x in b) == data and data == list(range(10)), "shuffle covers everything, data untouched")
check(sh == list(batches(data, 4, shuffle=True, seed=1)), "shuffle is reproducible with a seed")
check(list(sliding_windows([1, 2, 3, 4, 5], 3, 2)) == [[1, 2, 3], [3, 4, 5]], "sliding_windows")

✅ batches is a generator (uses yield)
✅ batches
✅ drop_last
✅ shuffle covers everything, data untouched
✅ shuffle is reproducible with a seed
✅ sliding_windows


## Part 5 — Recursion on a tree (20 min)

A fitted decision tree stored as nested dicts. Leaves are `{"value": v}`; internal nodes are
`{"feature": j, "threshold": t, "left": {...}, "right": {...}}` (go left if `x[j] <= t`).

In [10]:
tree = {"feature": 0, "threshold": 50,
        "left": {"value": 0.1},
        "right": {"feature": 1, "threshold": 1.5,
                  "left": {"value": 0.4},
                  "right": {"feature": 0, "threshold": 70, "left": {"value": 0.6}, "right": {"value": 0.9}}}}

def predict_one(node, x):
    """Recursive prediction for one sample x (a list)."""
    if "value" in node:
        return node["value"]
    return predict_one(node["left"] if x[node["feature"]] <= node["threshold"] else node["right"], x)

def depth(node):
    """Depth of the tree: a single leaf has depth 0."""
    return 0 if "value" in node else 1 + max(depth(node["left"]), depth(node["right"]))

def n_leaves(node):
    return 1 if "value" in node else n_leaves(node["left"]) + n_leaves(node["right"])

In [11]:
check([predict_one(tree, x) for x in ([30, 0], [60, 1], [60, 2], [80, 2])] == [0.1, 0.4, 0.6, 0.9], "predict_one")
check(depth(tree) == 3 and n_leaves(tree) == 4, "depth and n_leaves")

✅ predict_one
✅ depth and n_leaves


## Part 6 — Mini-project: a character bigram language model (40 min)

`P(next char | current char)` estimated by counting, with add-α smoothing. Sequences are wrapped with a boundary
character `"."` at the start and end, as in Karpathy's makemore — but this time from a blank page.

In [12]:
names = ["emma", "olivia", "ava", "isabella", "sophia", "mia", "amelia", "harper", "evelyn", "abigail",
         "emily", "ella", "elizabeth", "camila", "luna", "sofia", "avery", "mila", "aria", "scarlett"]

class BigramLM:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, words):
        """Build self.chars (sorted list of all characters incl. ".") and self.counts: dict of dict, counts[a][b]."""
        self.chars = sorted({c for w in words for c in w} | {"."})
        self.counts = {a: {b: 0 for b in self.chars} for a in self.chars}
        for w in words:
            s = "." + w + "."
            for a, b in zip(s, s[1:]):
                self.counts[a][b] += 1
        return self

    def prob(self, a, b):
        """Smoothed P(b | a) = (count + alpha) / (row total + alpha * V)."""
        V = len(self.chars)
        return (self.counts[a][b] + self.alpha) / (sum(self.counts[a].values()) + self.alpha * V)

    def nll(self, words):
        """Average negative log-likelihood per bigram over all words (boundaries included)."""
        total, n = 0.0, 0
        for w in words:
            s = "." + w + "."
            for a, b in zip(s, s[1:]):
                total -= math.log(self.prob(a, b)); n += 1
        return total / n

    def sample(self, n, seed=0):
        """Generate n words: start from ".", draw the next char with random.Random(seed).choices(chars, weights)
        until "." is drawn again."""
        rng = random.Random(seed); out = []
        for _ in range(n):
            a, w = ".", ""
            while True:
                b = rng.choices(self.chars, weights=[self.prob(a, c) for c in self.chars])[0]
                if b == ".":
                    break
                w += b; a = b
            out.append(w)
        return out

**✍️ Predict before running.** What is the NLL of a model that gives every character equal probability (V characters)? Will your trained model beat it on the training names? On a implausible string like 'bbbbb'?

> Prediction:
>
> Confidence: [1] / [2] / [3]

In [13]:
lm = BigramLM(alpha=1.0).fit(names)
V = len(lm.chars)
check(all(abs(sum(lm.prob(a, b) for b in lm.chars) - 1) < 1e-12 for a in lm.chars), "each row of P sums to 1")
check(lm.nll(names) < math.log(V), f"train NLL {lm.nll(names):.3f} < uniform log V = {math.log(V):.3f}")
check(lm.nll(["bbbbb"]) > lm.nll(names), "an implausible word gets higher NLL")
s = lm.sample(5, seed=0)
check(s == lm.sample(5, seed=0) and all(set(w) <= set(lm.chars) for w in s), f"sampling is reproducible: {s}")

✅ each row of P sums to 1
✅ train NLL 2.192 < uniform log V = 3.045
✅ an implausible word gets higher NLL
✅ sampling is reproducible: ['sriclisfllymfsofyztvesvpiacovzlvetn', 'oitr', 'ivelvcncztiaagy', 'lpmsnzoollhmgcbnpiarvyuvymipftuvmyniozyt', 'minufsa']


## Part 7 — Dataclasses and a patient-level split (20 min)

Prepares notebook 07: splitting by **patient**, never by row.

In [14]:
@dataclass
class Visit:
    patient_id: str
    day: int
    label: int

def patient_split(visits, test_frac=0.25, seed=0):
    """Split visits into (train, test) so that no patient appears in both.
    Sort the unique patient ids, shuffle them with random.Random(seed), the first round(test_frac * n_patients)
    go to test. Keep the visits' original order within each split."""
    ids = sorted({v.patient_id for v in visits})
    random.Random(seed).shuffle(ids)
    test_ids = set(ids[:round(test_frac * len(ids))])
    return [v for v in visits if v.patient_id not in test_ids], [v for v in visits if v.patient_id in test_ids]

In [15]:
rnd = random.Random(42)
visits = [Visit(f"p{rnd.randint(0, 19):02d}", d, rnd.randint(0, 1)) for d in range(200)]
tr, te = patient_split(visits)
ids_tr, ids_te = {v.patient_id for v in tr}, {v.patient_id for v in te}
check(not ids_tr & ids_te, "no patient in both splits")
check(len(tr) + len(te) == len(visits), "nothing lost")
check(len(ids_te) == round(0.25 * len(ids_tr | ids_te)), "25% of patients in test")
check([v.day for v in tr] == sorted(v.day for v in tr), "order preserved")

✅ no patient in both splits
✅ nothing lost
✅ 25% of patients in test
✅ order preserved


## Debrief → `learning/LOG.md`

- Time per part. Which part made you reach for help?
- Which Python features felt unfamiliar (comprehensions, `zip(*m)`, `yield`, dunder methods, `@dataclass`, recursion)?
  Those are your Python **still shaky** items.
- Rewrite `Vocab` from memory in a blank file in under 10 minutes. If you can't, redo Part 3 in three days.